In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/processed")
OUT = Path("../data/processed")

In [2]:
races = pd.read_csv(RAW / "races.csv")
results = pd.read_csv(RAW / "results.csv")
qualifying = pd.read_csv(RAW / "qualifying.csv")
lap_times = pd.read_csv(RAW / "lap_times.csv")
pit_stops = pd.read_csv(RAW / "pit_stops.csv")
drivers = pd.read_csv(RAW / "drivers.csv")
constructors = pd.read_csv(RAW / "constructors.csv")

In [3]:
for name, df in {
    "races": races,
    "results": results,
    "qualifying": qualifying,
    "lap_times": lap_times,
    "pit_stops": pit_stops
}.items():
    print(f"\n{name}")
    print(df.info())


races
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   raceId       1125 non-null   int64 
 1   year         1125 non-null   int64 
 2   round        1125 non-null   int64 
 3   circuitId    1125 non-null   int64 
 4   name         1125 non-null   object
 5   date         1125 non-null   object
 6   time         1125 non-null   object
 7   url          1125 non-null   object
 8   fp1_date     1125 non-null   object
 9   fp1_time     1125 non-null   object
 10  fp2_date     1125 non-null   object
 11  fp2_time     1125 non-null   object
 12  fp3_date     1125 non-null   object
 13  fp3_time     1125 non-null   object
 14  quali_date   1125 non-null   object
 15  quali_time   1125 non-null   object
 16  sprint_date  1125 non-null   object
 17  sprint_time  1125 non-null   object
dtypes: int64(4), object(14)
memory usage: 158.3+ KB
None

results

In [4]:
races_clean = races[[
    "raceId", "year", "round", "circuitId", "name"
]].rename(columns={"name": "raceName"})

In [5]:
results_clean = results[[
    "raceId", "driverId", "constructorId",
    "grid", "positionOrder", "points", "statusId"
]]

In [6]:
race_driver = results_clean.merge(
    races_clean,
    on="raceId",
    how="left"
)

In [7]:
qualifying_clean = qualifying[[
    "raceId", "driverId", "position"
]].rename(columns={"position": "qualifyingPosition"})

race_driver = race_driver.merge(
    qualifying_clean,
    on=["raceId", "driverId"],
    how="left"
)

In [8]:
lap_agg = (
    lap_times
    .groupby(["raceId", "driverId"])
    .agg(
        avgLapTime_ms=("milliseconds", "mean"),
        minLapTime_ms=("milliseconds", "min"),
        lapsCompleted=("lap", "count")
    )
    .reset_index()
)

race_driver = race_driver.merge(
    lap_agg,
    on=["raceId", "driverId"],
    how="left"
)

In [9]:
pit_agg = (
    pit_stops
    .groupby(["raceId", "driverId"])
    .agg(
        pitStopCount=("stop", "count"),
        avgPitDuration_ms=("milliseconds", "mean")
    )
    .reset_index()
)

race_driver = race_driver.merge(
    pit_agg,
    on=["raceId", "driverId"],
    how="left"
)

In [10]:
drivers_clean = drivers.loc[:, ["driverId", "forename", "surname"]].copy()
drivers_clean["driverName"] = (
    drivers_clean["forename"] + " " + drivers_clean["surname"]
)

race_driver = race_driver.merge(
    drivers_clean[["driverId", "driverName"]],
    on="driverId",
    how="left"
)

constructors_clean = constructors.loc[:, ["constructorId", "name"]].copy()
constructors_clean.rename(
    columns={"name": "constructorName"},
    inplace=True
)

race_driver = race_driver.merge(
    constructors_clean,
    on="constructorId",
    how="left"
)

In [11]:
race_driver["pitStopCount"] = race_driver["pitStopCount"].fillna(0)
race_driver["avgPitDuration_ms"] = race_driver["avgPitDuration_ms"].fillna(0)

In [12]:
print("Rows:", race_driver.shape[0])
print("Unique races:", race_driver["raceId"].nunique())
print("Unique drivers:", race_driver["driverId"].nunique())

race_driver.groupby("year")["driverId"].nunique().describe()

Rows: 26759
Unique races: 1125
Unique drivers: 861


count     75.000000
mean      42.813333
std       22.834122
min       20.000000
25%       25.000000
50%       36.000000
75%       51.000000
max      108.000000
Name: driverId, dtype: float64

In [13]:
race_driver.to_csv(
    OUT / "clean_race_driver.csv",
    index=False
)

In [14]:
print(race_driver.shape)
race_driver.head()

(26759, 19)


,raceId,driverId,constructorId,grid,positionOrder,points,statusId,year,round,circuitId,raceName,qualifyingPosition,avgLapTime_ms,minLapTime_ms,lapsCompleted,pitStopCount,avgPitDuration_ms,driverName,constructorName
0,18,1,1,1,1,10.0,1,2008,1,1,Australian Grand Prix,1.0,98114.068966,87452.0,58.0,0.0,0.0,Lewis Hamilton,McLaren
1,18,2,2,5,2,8.0,1,2008,1,1,Australian Grand Prix,5.0,98208.517241,87739.0,58.0,0.0,0.0,Nick Heidfeld,BMW Sauber
2,18,3,3,7,3,6.0,1,2008,1,1,Australian Grand Prix,7.0,98254.810345,88090.0,58.0,0.0,0.0,Nico Rosberg,Williams
3,18,4,4,11,4,5.0,1,2008,1,1,Australian Grand Prix,12.0,98410.293103,88603.0,58.0,0.0,0.0,Fernando Alonso,Renault
4,18,5,1,3,5,4.0,1,2008,1,1,Australian Grand Prix,3.0,98424.655172,87418.0,58.0,0.0,0.0,Heikki Kovalainen,McLaren
